# Pipeline Benchmark — DP Steel

Exhaustive grid search over every configurable stage of the prediction pipeline, restricted to **DP (dual-phase) steel** samples.

**Material constraint**: DP steel only (`alloy` contains 'DP' or 'dual-phase')  
**Targets**: `cycle1_holdingtemp_degc`, `cycle1_holdingtime_min`  
**Evaluation**: `RepeatedKFold(n_splits=5, n_repeats=10)` — more reliable than a single val split on this sample size

## Axes benchmarked

| Stage | Options |
|-------|---------|
| Column drop threshold | 0.50, 0.75, 0.90, 0.95 |
| Numeric imputation | mean, median, mice |
| Scaling | standard, minmax, robust, none |
| Categorical encoding | onehot, label |
| Image backbone | all cached + tabular-only |
| Regressor | RF, GBR, ABR + SVR, ExtraTrees, KNN |

Results are collected into a single DataFrame and ranked by mean CV R².

In [ ]:
import os, sys, gc, warnings, itertools, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('..'))

from src.column_sanitizer import sanitize_dataframe
from src.config import PreprocessingConfig, MissingDataConfig, ScalingConfig, EncodingConfig
from src.preprocessing import FeaturePreprocessor
from src.extraction.backbones import BackboneRegistry
from src import hyperparams as hp

from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    AdaBoostRegressor, ExtraTreesRegressor
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, make_scorer

SEED = 42
np.random.seed(SEED)

# Multi-output R² scorer: averages R² across both target columns
multi_r2 = make_scorer(r2_score, multioutput='uniform_average')

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
print('Setup complete.')
from src.features import FeaturePipeline

from src.run_store import RunStore
from src import metrics_viz
_str = str  # path helper


## 1. Load & Filter to DP Steel

In [ ]:
# ── RunStore: allocate this run's output directory ──────────────────────────
_store = RunStore("pipeline")
_run_dir, _run_id = _store.start()
print(f"Run directory: {_run_dir}")

df_raw = pd.read_csv('../datasets/metadata_latest.csv', header=1)
df_raw = sanitize_dataframe(df_raw)
df_raw = df_raw[df_raw['c'].notna()].copy().reset_index(drop=True)

# DP steel filter
dp_mask = df_raw['alloy'].str.lower().str.contains(r'dp|dual.?phase', na=False, regex=True)
df = df_raw[dp_mask].copy().reset_index(drop=True)

print(f'Full dataset:  {len(df_raw)} rows')
print(f'DP steel rows: {len(df)}')
print(f'Alloy variants: {df["alloy"].value_counts().to_dict()}')

# Target columns
C1_TEMP = 'cycle1_holdingtemp_degc'
C1_TIME = 'cycle1_holdingtime_min'
TARGET_COLS = [C1_TEMP, C1_TIME]

# Keep only rows with both targets
df = df[df[C1_TEMP].notna() & df[C1_TIME].notna()].copy().reset_index(drop=True)
print(f'Rows with both Cycle 1 targets: {len(df)}')

# Engineered feature
df['temp_time_ratio'] = df[C1_TEMP] / (df[C1_TIME] + 1e-6)

Y = df[TARGET_COLS].values.astype(np.float64)
print(f'Y shape: {Y.shape}')
# ── FeaturePipeline (read-only — caches built by prepare_features.ipynb) ───
fp = FeaturePipeline(
    data_dir=os.path.abspath('../data'),
    temp_dir=os.path.abspath('../data/temp_images'),
    features_dir=os.path.abspath('../features'),
)


## 2. Benchmark Configuration

In [ ]:
# ── Columns excluded from features (targets + leaky columns) ─────────────────
EXCLUDE_COLS = [
    'cycle1_holdingtemp_degc', 'cycle1_holdingtime_min',
    'cycle2_holdingtemp_degc', 'cycle2_holdingtime_min',
    'cycle3_holdingtemp_degc', 'cycle3_holdingtime_min',
    'cycle4_holdingtemp_degc', 'cycle4_holdingtime_min',
]
FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS]

COLUMN_TYPE_OVERRIDES = {
    'num_cycles':          'categorical',
    'cycle1_crate_degc_s': 'categorical',
    'cycle1_rtemp':        'categorical',
    'cycle1_qtemp':        'categorical',
    'heat_treatment_type': 'categorical',
    'id':                  'unique_string',
}
MICE_COLS      = ['cr', 'mo', 's', 'p', 'ni', 'al']
INDICATOR_COLS = ['ti', 'nb', 'v']

# ── Benchmark axes ────────────────────────────────────────────────────────────
DROP_THRESHOLDS    = [0.50, 0.75, 0.90, 0.95]
IMPUTE_STRATEGIES  = ['mean', 'median', 'mice']
SCALE_METHODS      = ['standard', 'minmax', 'robust', 'none']
ENCODE_METHODS     = ['onehot', 'label']

CV = RepeatedKFold(n_splits=5, n_repeats=10, random_state=SEED)
CACHE_DIR = os.path.abspath('../data')

print('Benchmark axes:')
print(f'  drop_threshold:   {DROP_THRESHOLDS}')
print(f'  impute_strategy:  {IMPUTE_STRATEGIES}')
print(f'  scale_method:     {SCALE_METHODS}')
print(f'  encode_method:    {ENCODE_METHODS}')
total_preproc = len(DROP_THRESHOLDS) * len(IMPUTE_STRATEGIES) * len(SCALE_METHODS) * len(ENCODE_METHODS)
print(f'  Total preprocessing combos: {total_preproc}')

## 3. Preprocessing Benchmark

Holds regressor fixed (RF, 100 trees) while sweeping all preprocessing options.
Uses `RepeatedKFold` CV — preprocessing is fit inside each fold to avoid leakage.

In [ ]:
from sklearn.model_selection import KFold

def make_preprocessor(drop_thresh, impute_strat, scale_method, encode_method):
    """Build a FeaturePreprocessor for a given config combo."""
    cfg = PreprocessingConfig(
        missing_data=MissingDataConfig(
            column_drop_threshold=drop_thresh,
            row_fill_threshold=0.50,
            numeric_fill_strategy=impute_strat if impute_strat != 'mice' else 'median',
            categorical_fill_strategy='mode',
            mice_max_iter=10,
        ),
        scaling=ScalingConfig(method=scale_method, enabled=(scale_method != 'none')),
        encoding=EncodingConfig(categorical=encode_method, max_categories=50),
    )
    active_overrides = {k: v for k, v in COLUMN_TYPE_OVERRIDES.items() if k in FEATURE_COLS}
    mice_cols        = MICE_COLS if impute_strat == 'mice' else []
    indicator_cols   = [c for c in INDICATOR_COLS if c in FEATURE_COLS]
    return FeaturePreprocessor(cfg, column_types=active_overrides,
                               mice_columns=mice_cols,
                               indicator_columns=indicator_cols)


def cv_r2_preprocessor(drop_thresh, impute_strat, scale_method, encode_method, n_splits=5, n_repeats=5):
    """CV R² with preprocessing fit inside each fold."""
    rf = RandomForestRegressor(n_estimators=100, max_features='sqrt',
                               min_samples_leaf=3, random_state=SEED, n_jobs=-1)
    cv = RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=SEED)
    scores = []

    for train_idx, test_idx in cv.split(df):
        df_tr = df.iloc[train_idx][FEATURE_COLS].copy()
        df_te = df.iloc[test_idx][FEATURE_COLS].copy()
        Y_tr  = Y[train_idx]
        Y_te  = Y[test_idx]

        try:
            prep = make_preprocessor(drop_thresh, impute_strat, scale_method, encode_method)
            X_tr = prep.fit_transform(df_tr)
            X_te = prep.transform(df_te)
            rf.fit(X_tr, Y_tr)
            score = r2_score(Y_te, rf.predict(X_te), multioutput='uniform_average')
            scores.append(score)
        except Exception:
            scores.append(np.nan)

    return np.nanmean(scores), np.nanstd(scores)


preproc_results = []
combos = list(itertools.product(DROP_THRESHOLDS, IMPUTE_STRATEGIES, SCALE_METHODS, ENCODE_METHODS))
print(f'Running {len(combos)} preprocessing combinations...')

for i, (dt, imp, sc, enc) in enumerate(combos):
    t0 = time.time()
    mean_r2, std_r2 = cv_r2_preprocessor(dt, imp, sc, enc)
    elapsed = time.time() - t0
    preproc_results.append({
        'drop_threshold': dt,
        'impute': imp,
        'scale': sc,
        'encode': enc,
        'mean_r2': mean_r2,
        'std_r2':  std_r2,
    })
    if (i + 1) % 8 == 0 or i == len(combos) - 1:
        print(f'  {i+1}/{len(combos)}  last: dt={dt} imp={imp} sc={sc} enc={enc}  '
              f'R²={mean_r2:.3f}±{std_r2:.3f}  ({elapsed:.1f}s)')

df_preproc = pd.DataFrame(preproc_results).sort_values('mean_r2', ascending=False).reset_index(drop=True)
print('\nTop 10 preprocessing configs:')
print(df_preproc.head(10).to_string(index=False))

In [ ]:
# ── Preprocessing results visualisation ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col, title in zip(
    axes.flat,
    ['impute', 'scale', 'encode', 'drop_threshold'],
    ['Imputation Strategy', 'Scaling Method', 'Categorical Encoding', 'Column Drop Threshold']
):
    grp = df_preproc.groupby(col)['mean_r2'].agg(['mean', 'std']).reset_index()
    grp = grp.sort_values('mean', ascending=False)
    ax.bar(grp[col].astype(str), grp['mean'], yerr=grp['std'],
           capsize=4, alpha=0.8, color='steelblue')
    ax.set_title(title)
    ax.set_ylabel('Mean CV R²')
    ax.set_ylim(max(0, grp['mean'].min() - 0.1), min(1, grp['mean'].max() + 0.1))

plt.suptitle('Preprocessing Stage Impact on CV R² (DP Steel, RF fixed)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(_str(_run_dir / 'preprocessing_benchmark.png'), dpi=150)
plt.show()

# Best config
best_preproc = df_preproc.iloc[0]
print(f"\nBest preprocessing config:")
print(f"  drop_threshold={best_preproc['drop_threshold']}, impute={best_preproc['impute']}, "
      f"scale={best_preproc['scale']}, encode={best_preproc['encode']}")
print(f"  CV R² = {best_preproc['mean_r2']:.4f} ± {best_preproc['std_r2']:.4f}")

## 4. Regressor Benchmark

Holds preprocessing fixed at the best config found above, sweeps all regressors.

In [ ]:
# ── Feature matrix (best preprocessing from §3) ──────────────────────────────
best_prep = make_preprocessor(
    best_preproc['drop_threshold'],
    best_preproc['impute'],
    best_preproc['scale'],
    best_preproc['encode'],
)
X_tab = best_prep.fit_transform(df[FEATURE_COLS].copy())
print(f'Feature matrix: {X_tab.shape}')

def multi_output(estimator):
    return MultiOutputRegressor(estimator, n_jobs=-1)

# ── Load saved hyperparams (written by bayes_tuning.ipynb) ────────────────────
_SCOPE       = 'dp_steel'
_saved       = hp.load(_SCOPE)
_best_saved  = None
if _saved:
    _best_saved_name, _best_saved_params = hp.best_model(_SCOPE)
    print(f'Loaded saved params for scope=\'{_SCOPE}\':')
    for name, entry in _saved.items():
        r2 = entry.get('tuned_cv_r2', entry.get('best_cv_r2', '?'))
        print(f'  {name:<12} tuned_cv_r2={r2}')
    print()
else:
    print(f'No saved hyperparams for scope=\'{_SCOPE}\' — using defaults.')
    _best_saved_name = None

# ── Build REGRESSORS dict — injects tuned params where available ──────────────
def _from_saved(name):
    """Return kwargs from saved params for model *name*, or None if not saved."""
    if not _saved or name not in _saved:
        return None
    return _saved[name].get('params')

def _rf_kwargs(name):
    p = _from_saved(name)
    if p:
        return {k: v for k, v in p.items()
                if k in ('n_estimators','max_depth','min_samples_leaf','max_features','bootstrap')}
    return dict(n_estimators=200, max_features='sqrt', min_samples_leaf=3)

def _gbr_kwargs():
    p = _from_saved('GBR')
    if p:
        return p
    return dict(n_estimators=200, learning_rate=0.05, max_depth=3,
                subsample=0.8, min_samples_leaf=5)

def _abr_kwargs():
    p = _from_saved('ABR')
    if p:
        md = p.pop('max_depth', 3)
        return {'base_depth': md, **p}
    return {'base_depth': 3, 'n_estimators': 100}

_abr_p = _abr_kwargs()
_abr_depth = _abr_p.pop('base_depth')

REGRESSORS = {
    'RF':         RandomForestRegressor(
                      **_rf_kwargs('RF'), random_state=SEED, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(
                      **_rf_kwargs('ExtraTrees'), random_state=SEED, n_jobs=-1),
    'GBR':        multi_output(GradientBoostingRegressor(
                      **_gbr_kwargs(), random_state=SEED)),
    'ABR':        multi_output(AdaBoostRegressor(
                      estimator=DecisionTreeRegressor(max_depth=_abr_depth),
                      **_abr_p, random_state=SEED)),
    'SVR_rbf':    multi_output(SVR(**({k:v for k,v in _from_saved('SVR_rbf').items()}
                                    if _from_saved('SVR_rbf') else
                                    dict(kernel='rbf', C=10, gamma='scale', epsilon=0.1)))),
    'SVR_linear': multi_output(SVR(**({k:v for k,v in _from_saved('SVR_linear').items()}
                                    if _from_saved('SVR_linear') else
                                    dict(kernel='linear', C=1, epsilon=0.1)))),
    'KNN_5':      multi_output(KNeighborsRegressor(n_neighbors=5)),
    'KNN_3':      multi_output(KNeighborsRegressor(n_neighbors=3)),
}

# ── Print which models use saved vs default params ────────────────────────────
print('Regressor configs:')
for name in REGRESSORS:
    src = 'saved \u2713' if _from_saved(name.replace('_5','').replace('_3','')) else 'default'
    print(f'  {name:<12} {src}')
print()

# ── Run CV sweep ──────────────────────────────────────────────────────────────
regressor_results = []
print(f'Running {len(REGRESSORS)} regressors with RepeatedKFold(5x10)...')
for name, model in REGRESSORS.items():
    t0     = time.time()
    scores = cross_val_score(model, X_tab, Y, cv=CV, scoring=multi_r2, n_jobs=-1)
    elapsed = time.time() - t0
    regressor_results.append({
        'regressor': name, 'mean_r2': scores.mean(), 'std_r2': scores.std(),
        'min_r2': scores.min(), 'max_r2': scores.max(),
    })
    marker = ' [saved]' if _from_saved(name.replace('_5','').replace('_3','')) else ''
    print(f'  {name:<12} R\u00b2={scores.mean():.4f}\u00b1{scores.std():.4f}  '
          f'[{scores.min():.3f}, {scores.max():.3f}]  ({elapsed:.1f}s){marker}')

df_reg = pd.DataFrame(regressor_results).sort_values('mean_r2', ascending=False).reset_index(drop=True)
print()
print('Regressor ranking:')
print(df_reg.to_string(index=False))

In [ ]:
# ── Regressor results visualisation ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
names  = df_reg['regressor'].tolist()
means  = df_reg['mean_r2'].tolist()
stds   = df_reg['std_r2'].tolist()
colors = ['#2ecc71' if m == max(means) else 'steelblue' for m in means]

ax.barh(names[::-1], means[::-1], xerr=stds[::-1], capsize=4, color=colors[::-1], alpha=0.85)
ax.set_xlabel('Mean CV R²')
ax.set_title('Regressor Comparison — DP Steel (best preprocessing, tabular features)', fontweight='bold')
ax.axvline(0, color='black', linewidth=0.5)
for i, (m, s) in enumerate(zip(means[::-1], stds[::-1])):
    ax.text(m + s + 0.005, i, f'{m:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(_str(_run_dir / 'regressor_benchmark.png'), dpi=150)
plt.show()

best_reg_name = df_reg.iloc[0]['regressor']
print(f'Best regressor: {best_reg_name}  R²={df_reg.iloc[0]["mean_r2"]:.4f}')

## 5. Image Backbone Benchmark

Holds preprocessing and regressor fixed at the best found above.  
Loads each available backbone cache, appends image features to tabular, reports CV R².

In [ ]:
# ── Image backbone benchmark ─────────────────────────────────────────────────
# CNN caches are built by prepare_features.ipynb. This cell only reads them.

from sklearn.base import clone

backbone_results = []
all_backbones = sorted(BackboneRegistry.list_available())

# Tabular-only baseline
tab_scores = cross_val_score(
    clone(REGRESSORS[best_reg_name]), X_tab, Y, cv=CV, scoring=multi_r2, n_jobs=-1
)
backbone_results.append({
    'backbone':  'tabular_only',
    'feat_dim':  X_tab.shape[1],
    'n_matched': len(df),
    'mean_r2':   tab_scores.mean(),
    'std_r2':    tab_scores.std(),
})
print(f'{"tabular_only":<22} feat={X_tab.shape[1]:>5}  R²={tab_scores.mean():.4f}±{tab_scores.std():.4f}')

for backbone_name in all_backbones:
    X_img = fp.load_image_features(backbone_name, df["id"])
    if X_img is None:
        print(f'{backbone_name:<22} no cache -- skipping')
        continue

    n_matched = int((~np.isnan(X_img).any(axis=1)).sum())
    X_combined = np.concatenate([X_img, X_tab], axis=1)

    t0     = time.time()
    scores = cross_val_score(
        clone(REGRESSORS[best_reg_name]), X_combined, Y,
        cv=CV, scoring=multi_r2, n_jobs=-1
    )
    elapsed = time.time() - t0

    backbone_results.append({
        'backbone':  backbone_name,
        'feat_dim':  X_combined.shape[1],
        'n_matched': n_matched,
        'mean_r2':   scores.mean(),
        'std_r2':    scores.std(),
    })
    print(f'{backbone_name:<22} feat={X_combined.shape[1]:>5}  '
          f'matched={n_matched}/{len(df)}  '
          f'R²={scores.mean():.4f}±{scores.std():.4f}  ({elapsed:.1f}s)')
    gc.collect()

df_backbone = (
    pd.DataFrame(backbone_results)
    .sort_values('mean_r2', ascending=False)
    .reset_index(drop=True)
)
print('\nBackbone ranking:')
print(df_backbone.to_string(index=False))


In [ ]:
# ── Backbone results visualisation ───────────────────────────────────────────
tab_baseline = df_backbone[df_backbone['backbone'] == 'tabular_only']['mean_r2'].values[0]
df_bb_img    = df_backbone[df_backbone['backbone'] != 'tabular_only'].copy()

fig, ax = plt.subplots(figsize=(11, 5))
names  = df_bb_img['backbone'].tolist()
means  = df_bb_img['mean_r2'].tolist()
stds   = df_bb_img['std_r2'].tolist()
colors = ['#2ecc71' if m > tab_baseline else '#e74c3c' for m in means]

x = range(len(names))
ax.bar(x, means, yerr=stds, capsize=4, color=colors, alpha=0.85)
ax.axhline(tab_baseline, color='navy', linestyle='--', linewidth=1.5, label=f'Tabular-only ({tab_baseline:.3f})')
ax.set_xticks(list(x))
ax.set_xticklabels(names, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Mean CV R²')
ax.set_title('Image Backbone Impact — DP Steel (best preproc + regressor)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(_str(_run_dir / 'backbone_benchmark.png'), dpi=150)
plt.show()

best_backbone = df_backbone.iloc[0]['backbone']
print(f'Best backbone: {best_backbone}  R²={df_backbone.iloc[0]["mean_r2"]:.4f}')
delta = df_backbone.iloc[0]['mean_r2'] - tab_baseline
print(f'Δ vs tabular-only: {delta:+.4f}')

## 6. Optimal Pipeline Summary

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import train_test_split

print('=' * 60)
print('OPTIMAL PIPELINE — DP STEEL')
print('=' * 60)
print(f'  Samples:           {len(df)} DP steel rows')
print(f'  drop_threshold:    {best_preproc["drop_threshold"]}')
print(f'  impute:            {best_preproc["impute"]}')
print(f'  scale:             {best_preproc["scale"]}')
print(f'  encode:            {best_preproc["encode"]}')
print(f'  regressor:         {best_reg_name}')
print(f'  image backbone:    {best_backbone}')
print()

if best_backbone != 'tabular_only':
    result = fp.load_image_features(best_backbone, df["id"])
    result = (result, None) if result is not None else None  # align return shape
    X_final = np.concatenate([result[0], X_tab], axis=1) if result is not None else X_tab
else:
    X_final = X_tab

# Final CV R²
final_scores = cross_val_score(clone(REGRESSORS[best_reg_name]), X_final, Y,
                               cv=CV, scoring=multi_r2, n_jobs=-1)
print(f'Final CV R²:  {final_scores.mean():.4f} ± {final_scores.std():.4f}')
print(f'              [min={final_scores.min():.3f}, max={final_scores.max():.3f}]')
print()

# Per-target breakdown on a held-out split
X_tr, X_te, Y_tr, Y_te = train_test_split(X_final, Y, test_size=0.2, random_state=SEED)
final_model = clone(REGRESSORS[best_reg_name])
final_model.fit(X_tr, Y_tr)
Y_pred = final_model.predict(X_te)
print('Per-target held-out results:')
for i, col in enumerate(TARGET_COLS):
    r2  = r2_score(Y_te[:, i], Y_pred[:, i])
    mae = np.mean(np.abs(Y_te[:, i] - Y_pred[:, i]))
    print(f'  {col}: R²={r2:.4f}, MAE={mae:.2f}')
print('=' * 60)

## 7. Bayesian Hyperparameter Optimisation

Runs [Optuna](https://optuna.org/) TPE (Tree-structured Parzen Estimator) on the **top-3 regressors** identified in §4, tuning each model’s hyperparameters while holding the best preprocessing config fixed.

**Design choices**:
- Preprocessing fitted inside every CV fold (no leakage)
- Objective: mean CV R² via `RepeatedKFold(5×5)` — faster than 5×10 for 100 trials
- 100 Optuna trials per model; TPE sampler with 20 random warm-up trials
- Search spaces are model-specific (log-uniform where values span orders of magnitude)
- Final winner re-evaluated with `RepeatedKFold(5×10)` for a fair comparison with §4
- Best params saved to `best_hyperparams.json` for reproducibility

In [ ]:
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Lighter CV for Optuna (5×5 instead of 5×10) ────────────────────────────────────────
CV_OPT = RepeatedKFold(n_splits=5, n_repeats=5, random_state=SEED)

# ── Top-3 regressors from the sweep ─────────────────────────────────────────────
TOP_N = 3
top_regressors = df_reg.head(TOP_N)['regressor'].tolist()
print(f'Tuning top-{TOP_N} regressors: {top_regressors}')
print(f'Feature matrix shape: {X_tab.shape}')
print()

# ── Search space definitions ──────────────────────────────────────────────────────────────────

def build_model(trial, name):
    """Construct a model from Optuna trial suggestions for the given regressor."""
    if name in ('RF', 'ExtraTrees'):
        p = dict(
            n_estimators    = trial.suggest_int('n_estimators', 50, 600),
            max_depth       = trial.suggest_int('max_depth', 3, 30),
            min_samples_leaf= trial.suggest_int('min_samples_leaf', 1, 10),
            max_features    = trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.3, 0.5]),
            bootstrap       = trial.suggest_categorical('bootstrap', [True, False]),
        )
        cls = RandomForestRegressor if name == 'RF' else ExtraTreesRegressor
        return cls(**p, random_state=SEED, n_jobs=-1)

    elif name == 'GBR':
        p = dict(
            n_estimators    = trial.suggest_int('n_estimators', 50, 500),
            learning_rate   = trial.suggest_float('learning_rate', 1e-3, 0.5, log=True),
            max_depth       = trial.suggest_int('max_depth', 2, 8),
            subsample       = trial.suggest_float('subsample', 0.4, 1.0),
            min_samples_leaf= trial.suggest_int('min_samples_leaf', 1, 20),
            max_features    = trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        )
        return multi_output(GradientBoostingRegressor(**p, random_state=SEED))

    elif name == 'ABR':
        max_depth = trial.suggest_int('max_depth', 1, 8)
        p = dict(
            n_estimators = trial.suggest_int('n_estimators', 50, 400),
            learning_rate= trial.suggest_float('learning_rate', 0.01, 2.0, log=True),
        )
        return multi_output(AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=max_depth), **p, random_state=SEED))

    elif name == 'SVR_rbf':
        p = dict(
            C      = trial.suggest_float('C', 0.01, 1000.0, log=True),
            gamma  = trial.suggest_categorical('gamma', ['scale', 'auto']),
            epsilon= trial.suggest_float('epsilon', 1e-3, 10.0, log=True),
        )
        return multi_output(SVR(kernel='rbf', **p))

    elif name == 'SVR_linear':
        p = dict(
            C      = trial.suggest_float('C', 0.01, 100.0, log=True),
            epsilon= trial.suggest_float('epsilon', 1e-3, 5.0, log=True),
        )
        return multi_output(SVR(kernel='linear', **p))

    elif name.startswith('KNN'):
        p = dict(
            n_neighbors= trial.suggest_int('n_neighbors', 2, min(15, len(df) // 5)),
            weights    = trial.suggest_categorical('weights', ['uniform', 'distance']),
            p          = trial.suggest_int('p', 1, 2),  # 1=Manhattan 2=Euclidean
        )
        return multi_output(KNeighborsRegressor(**p))

    raise ValueError(f'No search space defined for: {name}')


def make_objective(reg_name):
    """Return an Optuna objective function closed over reg_name."""
    def objective(trial):
        model  = build_model(trial, reg_name)
        scores = cross_val_score(model, X_tab, Y, cv=CV_OPT,
                                 scoring=multi_r2, n_jobs=-1)
        return float(scores.mean())
    return objective


# ── Run optimisation ──────────────────────────────────────────────────────────────────
N_TRIALS      = 100
bayes_results = {}

for reg_name in top_regressors:
    print(f'Optimising {reg_name} ({N_TRIALS} trials)...')
    t0 = time.time()

    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=SEED, n_startup_trials=20),
        pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=5),
    )
    study.optimize(make_objective(reg_name), n_trials=N_TRIALS, show_progress_bar=False)

    elapsed = time.time() - t0
    print(f'  best CV R² = {study.best_value:.4f}  ({elapsed:.0f}s)')
    print(f'  params    = {study.best_params}')
    bayes_results[reg_name] = {
        'study':       study,
        'best_r2_opt': study.best_value,
        'best_params': study.best_params,
    }
    print()


# ── Helper: reconstruct model from a plain params dict ─────────────────────────
class _FixedTrial:
    """Minimal Optuna-compatible trial returning pre-set params."""
    def __init__(self, params):              self._p = params
    def suggest_int(self, n, *a, **kw):      return self._p[n]
    def suggest_float(self, n, *a, **kw):    return self._p[n]
    def suggest_categorical(self, n, *a, **kw): return self._p[n]


# ── Re-evaluate with full RepeatedKFold(5×10) ────────────────────────────────────────
print('Re-evaluating tuned models with RepeatedKFold(5×10)...')
print('─' * 60)

tuned_eval = []
for reg_name in top_regressors:
    res         = bayes_results[reg_name]
    best_model  = build_model(_FixedTrial(res['best_params']), reg_name)
    full_scores = cross_val_score(best_model, X_tab, Y, cv=CV,
                                  scoring=multi_r2, n_jobs=-1)
    baseline    = df_reg.loc[df_reg['regressor'] == reg_name, 'mean_r2'].values[0]
    delta       = full_scores.mean() - baseline
    tuned_eval.append({
        'regressor':  reg_name,
        'untuned_r2': baseline,
        'tuned_r2':   full_scores.mean(),
        'tuned_std':  full_scores.std(),
        'delta':      delta,
    })
    print(f'  {reg_name:<12}  untuned={baseline:.4f}  '
          f'tuned={full_scores.mean():.4f}±{full_scores.std():.4f}  '
          f'Δ={delta:+.4f}')

df_tuned = (pd.DataFrame(tuned_eval)
              .sort_values('tuned_r2', ascending=False)
              .reset_index(drop=True))
best_tuned_name   = df_tuned.iloc[0]['regressor']
best_tuned_r2     = df_tuned.iloc[0]['tuned_r2']
best_tuned_params = bayes_results[best_tuned_name]['best_params']
print()
print(f'Best tuned model: {best_tuned_name}  CV R²={best_tuned_r2:.4f}')
print(f'Improvement over untuned: {df_tuned.iloc[0]["delta"]:+.4f}')

In [ ]:
# ── Visualisation: tuned vs untuned + optimisation history ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left — grouped bar: untuned vs tuned
ax = axes[0]
names   = df_tuned['regressor'].tolist()
x       = np.arange(len(names))
w       = 0.35
untuned = df_tuned['untuned_r2'].tolist()
tuned   = df_tuned['tuned_r2'].tolist()
errs    = df_tuned['tuned_std'].tolist()

ax.bar(x - w/2, untuned, w, label='Untuned (§4)', alpha=0.7, color='steelblue')
ax.bar(x + w/2, tuned,   w, label='Bayesian-tuned', alpha=0.85, color='#2ecc71',
       yerr=errs, capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylabel('Mean CV R²')
ax.set_title('Hyperparameter Tuning Impact', fontweight='bold')
ax.legend()

for xi, (u, t, e) in enumerate(zip(untuned, tuned, errs)):
    col = '#27ae60' if (t - u) >= 0 else '#e74c3c'
    ax.text(xi + w/2, t + e + 0.005, f'{t - u:+.3f}',
            ha='center', va='bottom', fontsize=8, color=col, fontweight='bold')

# Right — optimisation history (best-so-far curve)
ax2    = axes[1]
cmap   = plt.cm.tab10.colors
for ci, reg_name in enumerate(top_regressors):
    study = bayes_results[reg_name]['study']
    vals  = [t.value for t in study.trials if t.value is not None]
    ax2.plot(range(1, len(vals) + 1), np.maximum.accumulate(vals),
             label=reg_name, color=cmap[ci], linewidth=1.8)

ax2.set_xlabel('Trial')
ax2.set_ylabel('Best CV R² so far')
ax2.set_title('Optuna Optimisation History', fontweight='bold')
ax2.legend()

plt.suptitle('Bayesian Hyperparameter Optimisation — DP Steel',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(_str(_run_dir / 'bayes_tuning.png'), dpi=150)
plt.show()

In [ ]:
# ── Best parameters table ──────────────────────────────────────────────────────────────────
print('Best hyperparameters found by Optuna:')
print('=' * 60)
for reg_name in top_regressors:
    res = bayes_results[reg_name]
    print(f'\n{reg_name}  (CV R²={res["best_r2_opt"]:.4f})')
    for k, v in res['best_params'].items():
        print(f'  {k:<24} = {v}')

# ── Parameter importance via Optuna FAnova ─────────────────────────────────
print('\nParameter importance (FAnova):')
print('─' * 60)
for reg_name in top_regressors:
    try:
        imp = optuna.importance.get_param_importances(bayes_results[reg_name]['study'])
        print(f'\n{reg_name}:')
        for k, v in imp.items():
            print(f'  {k:<24} {v:.3f}  {"█" * int(v * 30)}')
    except Exception as e:
        print(f'  {reg_name}: importance unavailable ({e})')

# ── Persist best params to JSON ──────────────────────────────────────────────────────────────────
# ── Persist to canonical hyperparameter store ─────────────────────────────────
_SCOPE = 'dp_steel'
_models_out = {
    name: {
        'best_cv_r2':  bayes_results[name]['best_r2_opt'],
        'tuned_cv_r2': float(df_tuned.loc[df_tuned['regressor']==name, 'tuned_r2'].values[0]),
        'tuned_std':   float(df_tuned.loc[df_tuned['regressor']==name, 'tuned_std'].values[0]),
        'delta':       float(df_tuned.loc[df_tuned['regressor']==name, 'delta'].values[0]),
        'params':      bayes_results[name]['best_params'],
    }
    for name in top_regressors
}
store_path = hp.save(
    scope=_SCOPE,
    models_dict=_models_out,
    n_trials=N_TRIALS,
    cv_protocol='RepeatedKFold(n_splits=5, n_repeats=5) search / (n_repeats=10) eval',
    feature_matrix_shape=X_tab.shape,
)
print(f'\nSaved to: {store_path}')
print()
print(hp.summary())

In [ ]:
# ── Full results table (all stages + tuned) ─────────────────────────────────────────────
df_preproc['stage']  = 'preprocessing'
df_reg['stage']      = 'regressor'
df_backbone['stage'] = 'backbone'
df_tuned['stage']    = 'tuned'

df_all = pd.concat([
    df_preproc.assign(config=df_preproc.apply(
        lambda r: f"dt={r['drop_threshold']} imp={r['impute']} sc={r['scale']} enc={r['encode']}", axis=1)),
    df_reg.rename(columns={'regressor': 'config'}),
    df_backbone.rename(columns={'backbone': 'config'}),
    df_tuned.rename(columns={'regressor': 'config', 'tuned_r2': 'mean_r2', 'tuned_std': 'std_r2'}),
], ignore_index=True)[['stage', 'config', 'mean_r2', 'std_r2']]

# Save CSV to both the run directory and the legacy location
_csv_path = _run_dir / 'benchmark_results.csv'
df_all.to_csv(str(_csv_path), index=False)
df_all.to_csv('benchmark_results.csv', index=False)  # legacy path for load-from-store
print(f'Saved benchmark_results.csv -> {_csv_path}')

# ── Copy visualisation artifacts to run dir ───────────────────────────────────
# (already saved there via redirected savefig calls above)

# ── Write manifest ───────────────────────────────────────────────────────────
_best_tuned_row = df_tuned.iloc[0] if len(df_tuned) else {}
_store.write_manifest({
    'scope':            'dp_steel',
    'n_samples':        len(df),
    'n_features':       int(X_tab.shape[1]),
    'best_preproc':     best_preproc,
    'best_regressor':   best_reg_name,
    'best_backbone':    best_backbone,
    'best_preproc_r2':  float(df_preproc.iloc[0]['mean_r2']),
    'best_reg_r2':      float(df_reg.iloc[0]['mean_r2']),
    'best_backbone_r2': float(df_backbone.iloc[0]['mean_r2']),
    'best_tuned_r2':    float(_best_tuned_row.get('tuned_r2', float('nan'))),
    'n_trials':         N_TRIALS,
})

# ── Append to history ─────────────────────────────────────────────────────────
_store.append_history({
    'n_samples':        len(df),
    'n_features':       int(X_tab.shape[1]),
    'best_preproc_r2':  float(df_preproc.iloc[0]['mean_r2']),
    'best_reg_r2':      float(df_reg.iloc[0]['mean_r2']),
    'best_backbone_r2': float(df_backbone.iloc[0]['mean_r2']),
    'best_tuned_r2':    float(_best_tuned_row.get('tuned_r2', float('nan'))),
    'best_model':       best_reg_name,
    'best_backbone':    best_backbone,
    'n_trials':         N_TRIALS,
})

# ── Regenerate history visualisation ─────────────────────────────────────────
metrics_viz.plot_history('pipeline', save=True, show=False)
print(f'History visualisation -> runs/pipeline/metrics_history.png')
print(f'Run artifacts stored  -> {_run_dir}')

print()
print(df_all.groupby('stage').apply(lambda x: x.nlargest(3, 'mean_r2')).to_string())

In [ ]:
# ── Heatmap: preprocessing config vs mean R² ─────────────────────────────────
pivot = df_preproc.pivot_table(
    values='mean_r2',
    index=['impute', 'scale'],
    columns=['drop_threshold', 'encode'],
    aggfunc='mean'
)
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=pivot.values.min(), vmax=pivot.values.max(),
            linewidths=0.5, ax=ax)
ax.set_title('Mean CV R² by Preprocessing Config — DP Steel', fontweight='bold')
plt.tight_layout()
plt.savefig('preprocessing_heatmap.png', dpi=150)
plt.show()